# Tester l'OCR sur les zones détectées par le modèle `champion`

**Quand ?** Jour 2, une fois le modèle `champion` de votre squad enregistré dans MLflow.

**Quoi ?** Ce notebook :

1. télécharge le modèle ONNX `flexistaff_<squad>@champion` depuis le registre MLflow ;
2. détecte les zones (avec **vos** classes) sur quelques images ;
3. lance EasyOCR sur chaque zone détectée, avec les mêmes réglages que l'application CalEvent ;
4. résume ce qui est lu, classe par classe, et sauvegarde le tout dans `ocr_<alias>.json`.

**Pourquoi ?** Pour préparer le **parsing** du jour 3 : voir concrètement ce que l'OCR lit dans vos zones,
quelles classes portent la date, l'heure, le titre, et quelles erreurs il faudra corriger.

**Prérequis**

- le fichier `identifiants-<login>.json` téléchargé depuis le portail (à déposer dans Colab : panneau *Fichiers*) ;
- un modèle enregistré sous `flexistaff_<squad>` avec l'alias `champion` ;
- rien d'autre : le jeu d'images de test commun (photos de plannings jamais vues à l'entraînement) est
  téléchargé automatiquement depuis la release GitHub du projet. Vous pouvez aussi tester vos propres images.

> Sur Colab, activez un GPU (*Exécution → Modifier le type d'exécution*) : EasyOCR sera beaucoup plus rapide.

## 1. Installation

In [ ]:
%pip install -q ultralytics mlflow onnx onnxruntime onnxslim easyocr

## 2. Paramètres

Adaptez ces valeurs à votre squad. `IMAGES` peut être un dossier **ou** une seule image.

In [ ]:
CREDENTIALS = "identifiants-<login>.json"  # fichier téléchargé depuis le portail
SQUAD = "team-rocket"                      # team-rocket, team-magma ou team-aqua
ALIAS = "champion"                         # puis "challenger" pour comparer
IMAGES = "test_ocr"                        # jeu de test commun (téléchargé ci-dessous) ; ou votre dossier / image
CONF = 0.25                                # seuil de confiance de la détection YOLO
MAX_IMAGES = 10                            # nombre d'images traitées (OCR lent sur CPU) ; None = toutes, avec un GPU
OCR_MIN_CONF = 0.4                         # lignes OCR sous ce seuil ignorées (comme dans CalEvent)
CLASSES_OCR = None                         # None = OCR sur toutes vos classes ; ou une liste, ex. ["event"]

MODEL_NAME = f"flexistaff_{SQUAD}"

# Jeu de test commun : le même pour toutes les squads, jamais utilisé pour l'entraînement
TEST_ZIP_URL = "https://github.com/CassiaFa/flexistaff/releases/download/test-ocr-v1/test_ocr.zip"

### Jeu d'images de test

Le jeu commun est téléchargé une seule fois (dossier `test_ocr/`). Comparer les modèles sur les **mêmes** images, jamais vues à l'entraînement, rend la comparaison champion / challenger (et entre squads) équitable.

In [ ]:
import urllib.request, zipfile
from pathlib import Path

if IMAGES == "test_ocr" and not Path("test_ocr").is_dir():
    urllib.request.urlretrieve(TEST_ZIP_URL, "test_ocr.zip")
    with zipfile.ZipFile("test_ocr.zip") as z:
        z.extractall(".")
print(f"Images : {IMAGES} ({len(list(Path(IMAGES).glob('*')))} fichiers)")


## 3. Connexion à MLflow et téléchargement du modèle

Les identifiants sont lus dans le fichier JSON et passés par variables d'environnement :
ils ne sont **jamais** affichés.

In [ ]:
import json
import os
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient

with open(CREDENTIALS, encoding="utf-8") as f:
    creds = json.load(f)["mlflow"]
os.environ["MLFLOW_TRACKING_USERNAME"] = creds["username"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = creds["password"]
mlflow.set_tracking_uri(creds["tracking_uri"])

# Quelle version porte l'alias, et de quel run vient-elle ?
client = MlflowClient()
version = client.get_model_version_by_alias(MODEL_NAME, ALIAS)
run_name = client.get_run(version.run_id).info.run_name if version.run_id else "?"
print(f"Modèle : {MODEL_NAME}@{ALIAS} → version {version.version} (run « {run_name} »)")

# Téléchargement : un dossier qui contient model.onnx
model_dir = Path(f"modele_{ALIAS}")
mlflow.artifacts.download_artifacts(f"models:/{MODEL_NAME}@{ALIAS}", dst_path=str(model_dir))
onnx_files = sorted(model_dir.rglob("*.onnx"))
if not onnx_files:
    raise FileNotFoundError(f"Aucun fichier .onnx trouvé dans {model_dir}")
onnx_path = onnx_files[0]
print("Fichier ONNX :", onnx_path)

## 4. Chargement du modèle et de ses classes

Ultralytics enregistre les noms des classes dans les métadonnées du fichier ONNX.

In [ ]:
from ultralytics import YOLO

model = YOLO(str(onnx_path), task="detect")
# model.names n'est rempli qu'au premier predict pour un modèle ONNX : on lance une
# prédiction « à blanc » sur une image noire pour charger les métadonnées.
import numpy as np
model.predict(np.zeros((640, 640, 3), dtype=np.uint8), imgsz=640, verbose=False)

print("Classes du modèle :")
for idx, nom in model.names.items():
    print(f"  {idx} : {nom}")

if all(str(nom).startswith("class") for nom in model.names.values()):
    print("\n⚠ Noms génériques (class0, class1…) : les métadonnées sont absentes du fichier ONNX.\n"
          "  Retrouvez les vrais noms dans le data.yaml de votre dataset (clé « names »).")

## 5. Lecteur OCR (mêmes réglages que l'application CalEvent)

On reproduit les réglages de `calevent/backend/app/services/ocr.py` :

- langues `fr` et `en` ;
- prétraitement de chaque zone : niveaux de gris, agrandissement si la zone fait moins de 64 px de haut,
  marge de 8 px autour (pour ne pas couper le texte collé au bord) ;
- paramètres de `readtext` abaissés pour ne pas rater les petits numéros de jour ;
- lignes de confiance inférieure à `OCR_MIN_CONF` ignorées (et comptées), puis tri dans l'ordre de lecture.

Différence avec l'application : CalEvent ne lit que certaines de **ses** classes. Ici, vous avez vos propres
classes, donc l'OCR est lancé sur **toutes** les zones détectées (sauf si vous restreignez avec `CLASSES_OCR`).

Le premier lancement télécharge les modèles EasyOCR (~100 Mo).

In [ ]:
import cv2
import easyocr
import torch

MIN_CROP_HEIGHT = 64  # hauteur minimale d'une zone avant OCR (px)
PADDING = 8           # marge ajoutée autour de la zone (px)
READTEXT_KWARGS = {"mag_ratio": 2, "min_size": 5, "low_text": 0.3, "text_threshold": 0.5}

gpu = torch.cuda.is_available()
reader = easyocr.Reader(["fr", "en"], gpu=gpu, verbose=False)
print("EasyOCR prêt, GPU :", gpu)


def pretraiter(crop):
    # Niveaux de gris + agrandissement des petites zones + marge
    gris = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop
    h = gris.shape[0]
    if h < MIN_CROP_HEIGHT:
        echelle = MIN_CROP_HEIGHT / h
        gris = cv2.resize(gris, None, fx=echelle, fy=echelle, interpolation=cv2.INTER_CUBIC)
    return cv2.copyMakeBorder(gris, PADDING, PADDING, PADDING, PADDING, cv2.BORDER_REPLICATE)


def lire_zone(crop):
    # Renvoie (texte, confiance moyenne, nb de lignes ignorées) ; texte "" et confiance None si rien n'est lu
    if crop.size == 0:
        return "", None, 0
    resultats = reader.readtext(pretraiter(crop), **READTEXT_KWARGS)
    # resultats : liste de (4 points du polygone, texte, confiance)
    lignes = []
    nb_ignorees = 0
    for points, texte, conf in resultats:
        texte = texte.strip()
        if not texte:
            continue
        if conf < OCR_MIN_CONF:
            nb_ignorees += 1  # ligne lue, mais trop peu sûre
            continue
        ys = [p[1] for p in points]
        xs = [p[0] for p in points]
        lignes.append({"texte": texte, "conf": float(conf),
                       "cy": (min(ys) + max(ys)) / 2, "h": max(ys) - min(ys), "x": min(xs)})
    if not lignes:
        return "", None, nb_ignorees
    # Ordre de lecture : on regroupe par ligne visuelle (haut → bas), puis gauche → droite
    lignes.sort(key=lambda l: l["cy"])
    h_moy = sum(l["h"] for l in lignes) / len(lignes)
    rangees = [[lignes[0]]]
    for l in lignes[1:]:
        if abs(l["cy"] - rangees[-1][-1]["cy"]) <= h_moy / 2:
            rangees[-1].append(l)
        else:
            rangees.append([l])
    ordre = [l for r in rangees for l in sorted(r, key=lambda l: l["x"])]
    texte = " ".join(l["texte"] for l in ordre)
    conf = sum(l["conf"] for l in ordre) / len(ordre)
    return texte, round(conf, 3), nb_ignorees

## 6. Détection + OCR sur les images

Pour chaque image : détection YOLO, découpe de chaque zone, OCR, puis affichage de l'image annotée
et d'un tableau des zones.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
chemin = Path(IMAGES)
if chemin.is_dir():
    images = sorted(p for p in chemin.iterdir() if p.suffix.lower() in EXTENSIONS)
else:
    images = [chemin]
images = images[:MAX_IMAGES]
print(f"{len(images)} image(s) à traiter")

toutes_les_lignes = []  # une ligne par zone détectée, toutes images confondues
resultats_json = []     # même contenu, regroupé par image (pour le fichier JSON)

for image_path in images:
    result = model.predict(str(image_path), conf=CONF, imgsz=640, max_det=1000, verbose=False)[0]
    img = result.orig_img  # image d'origine en BGR (numpy)
    h_img, w_img = img.shape[:2]

    lignes = []
    for box in result.boxes:
        x1, y1, x2, y2 = (int(v) for v in box.xyxy[0].tolist())
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w_img, x2), min(h_img, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        classe = model.names[int(box.cls)]
        if CLASSES_OCR is None or classe in CLASSES_OCR:
            # Découpe exacte de la boîte (bornée à l'image), comme dans l'application
            texte, conf_ocr, nb_ignorees = lire_zone(img[y1:y2, x1:x2])
        else:
            texte, conf_ocr, nb_ignorees = "", None, 0  # classe exclue de l'OCR
        lignes.append({
            "image": image_path.name,
            "classe": classe,
            "confiance_detection": round(float(box.conf), 3),
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "texte": texte,
            "confiance_ocr": conf_ocr,
            "lignes_ignorees": nb_ignorees,
        })

    # Affichage : image annotée (plot() renvoie du BGR → conversion en RGB)
    plt.figure(figsize=(12, 9))
    plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
    plt.title(f"{image_path.name} : {len(lignes)} zone(s)")
    plt.axis("off")
    plt.show()

    if lignes:
        df_image = pd.DataFrame(lignes).sort_values(["y1", "x1"])
        display(df_image.drop(columns="image"))
    else:
        print("Aucune zone détectée : essayez de baisser CONF.")

    toutes_les_lignes.extend(lignes)
    resultats_json.append({"image": image_path.name, "largeur": w_img, "hauteur": h_img,
                           "zones": [{k: v for k, v in l.items() if k != "image"} for l in lignes]})

## 7. Résumé par classe

Pour chaque classe : nombre de zones, part des zones où l'OCR n'a rien lu, confiance OCR moyenne,
nombre de lignes ignorées (confiance < `OCR_MIN_CONF`) et quelques exemples de textes lus.
Beaucoup de lignes ignorées ? Essayez de baisser `OCR_MIN_CONF` pour voir ce que l'OCR « devine ».

In [ ]:
df = pd.DataFrame(toutes_les_lignes)
if df.empty:
    print("Aucune zone détectée sur ces images.")
else:
    resume = df.groupby("classe").agg(
        nb_zones=("texte", "size"),
        part_texte_vide=("texte", lambda s: round((s == "").mean(), 2)),
        confiance_ocr_moyenne=("confiance_ocr", lambda s: round(s.mean(), 3)),
        lignes_ignorees=("lignes_ignorees", "sum"),
        exemples=("texte", lambda s: [t for t in s if t][:3]),
    )
    display(resume)
    print(f"Lignes OCR ignorées (confiance < {OCR_MIN_CONF}) : {int(df['lignes_ignorees'].sum())}")

## 8. Sauvegarde des résultats

Le fichier `ocr_<alias>.json` contient, image par image, les zones détectées et le texte lu.
Téléchargez-le (panneau *Fichiers* de Colab) : il vous servira à écrire et tester votre parser
**sans relancer** le modèle ni l'OCR.

In [ ]:
sortie = Path(f"ocr_{ALIAS}.json")
with open(sortie, "w", encoding="utf-8") as f:
    json.dump({"modele": f"{MODEL_NAME}@{ALIAS}", "version": version.version,
               "classes": {int(k): v for k, v in model.names.items()},
               "images": resultats_json}, f, ensure_ascii=False, indent=2)
print("Résultats sauvegardés dans", sortie)

## 9. Pistes pour le parsing

Pas de solution ici : des questions pour préparer le jour 3. Notez vos réponses dans vos livrables.

- **Qui porte quoi ?** Avec **vos** classes, laquelle contient la date (le numéro du jour, le mois) ?
  l'heure ? le titre de l'événement ? Certaines classes sont-elles inutiles pour le parsing ?
- **Relier un événement à son jour** : comment savoir à quel jour appartient une zone « événement » ?
  Regardez les coordonnées `x1, y1, x2, y2` : une zone est-elle *contenue* dans une autre ?
  Sur la même ligne, la même colonne qu'un en-tête ?
- **Formats d'heure** : quels formats apparaissent dans vos images (`8h30`, `08:30`, `8h-12h`, `14h00 - 18h`…) ?
  Quelle expression régulière les couvrirait tous ?
- **Confusions de l'OCR** : repérez-vous des lettres à la place de chiffres (`O` → `0`, `I`/`l` → `1`, `S` → `5`) ?
  Dans quels cas peut-on les corriger sans risque ?
- **Texte vide ou peu sûr** : que faire d'une zone sans texte, ou lue avec une confiance faible ?
  L'application CalEvent attribue une confiance à chaque événement extrait et l'utilise ainsi :
  **≥ 0,8 pré-coché**, **0,5 – 0,8 à vérifier**, **< 0,5 non proposé**.
  Comment calculeriez-vous cette confiance (détection, OCR, parsing réussi ou non) ?
- **Rappel avant tout** : une zone ratée par le modèle, c'est un créneau perdu. Quelles zones manquent
  sur les images ci-dessus ?

## 10. Comparer avec le challenger

Au jour 2, une fois un modèle `challenger` enregistré :

1. remplacez `ALIAS = "champion"` par `ALIAS = "challenger"` dans les paramètres ;
2. relancez tout le notebook (*Exécution → Tout exécuter*) : les résultats vont dans `ocr_challenger.json` ;
3. comparez les deux résumés par classe : nombre de zones, part de textes vides, confiance OCR.

Critère de promotion : **le rappel d'abord**, la mAP50 pour départager, et un coup d'œil à l'OCR
(un modèle qui découpe mieux les zones donne un texte plus propre). C'est le ou la **lead** de la squad
qui décide de déplacer l'alias `champion`.

## 11. Bonus : `best.pt` ou `model.onnx` ? (sur CPU)

Votre run contient deux versions du même modèle : `best.pt` (PyTorch, pour entraîner) et `model.onnx`
(pour déployer). CalEvent sert l'**ONNX sur CPU, sans PyTorch**. Est-ce vraiment mieux ? On mesure, au lieu de le croire :

| Mesure | Pourquoi |
|---|---|
| **taille du fichier** | à télécharger, à embarquer dans une app mobile |
| **temps de chargement** | démarrage du service, rechargement à chaud (↻) |
| **temps d'inférence** (médiane, ms / image) | ce que l'utilisateur attend |
| **mémoire maximale** du processus | ce que coûte le serveur (la VM est partagée) |
| **mêmes détections ?** | l'export ne doit rien changer au résultat |

Trois façons de lancer le modèle, chacune dans **son propre processus** Python (sinon la mémoire de l'une
pollue la mesure de l'autre) :

- `pt` : `best.pt` avec Ultralytics (PyTorch) ;
- `onnx` : `model.onnx` avec Ultralytics (onnxruntime derrière, mais PyTorch est quand même importé) ;
- `ort` : `model.onnx` avec **onnxruntime seul**, comme dans CalEvent (ni PyTorch ni Ultralytics ;
  on mesure l'inférence brute, sans le post-traitement).

Tout tourne sur **CPU** (même si Colab a un GPU) : c'est la situation du serveur de déploiement.
Les chiffres dépendent de la machine (un Colab gratuit a 2 cœurs) : comparez-les **entre eux**, sur la même machine.


In [ ]:
# Récupère best.pt dans le run de la version testée (il y a été loggé avec mlflow.log_artifact)
pt_path = Path(mlflow.artifacts.download_artifacts(run_id=version.run_id, artifact_path="best.pt",
                                                   dst_path=f"modele_{ALIAS}_pt"))
print("best.pt :", pt_path)

N_REPET = 5            # passages par image (le premier, à froid, est écarté)
images_bench = [str(p) for p in images[:5]]  # 5 images suffisent pour un ordre de grandeur


In [ ]:
import subprocess
import sys

# Script de mesure, lancé dans un processus neuf pour chaque façon de faire
Path("mesure_inference.py").write_text(r"""
import json, resource, sys, time
mode, chemin, n_repet, conf = sys.argv[1], sys.argv[2], int(sys.argv[3]), float(sys.argv[4])
images = json.loads(sys.argv[5])
import cv2
import numpy as np

t0 = time.perf_counter()
if mode == "ort":
    import onnxruntime as ort
    sess = ort.InferenceSession(chemin, providers=["CPUExecutionProvider"])
    entree = sess.get_inputs()[0]
    h, w = (d if isinstance(d, int) else 640 for d in entree.shape[2:4])
    def inferer(p):
        img = cv2.resize(cv2.imread(p), (w, h))[:, :, ::-1]  # BGR -> RGB
        x = np.ascontiguousarray(img.transpose(2, 0, 1)[None], dtype=np.float32) / 255.0
        sess.run(None, {entree.name: x})
        return None  # sortie brute : pas de NMS, donc pas de nombre de boîtes
else:
    from ultralytics import YOLO
    model = YOLO(chemin, task="detect")
    def inferer(p):
        r = model.predict(p, conf=conf, imgsz=640, max_det=1000, device="cpu", verbose=False)[0]
        return len(r.boxes)
    inferer(images[0])  # premier appel : charge réellement le modèle (ONNX)
chargement = time.perf_counter() - t0

durees, boites = [], []
for p in images:
    boites.append(inferer(p))          # passage à froid, écarté
    for _ in range(n_repet):
        t = time.perf_counter()
        inferer(p)
        durees.append((time.perf_counter() - t) * 1000)

print(json.dumps({
    "chargement_s": round(chargement, 2),
    "inference_ms_mediane": round(float(np.median(durees)), 1),
    "memoire_max_mo": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024),  # Linux : ko
    "boites": boites,
}))
""")

mesures = {}
for mode, chemin in [("pt", pt_path), ("onnx", onnx_path), ("ort", onnx_path)]:
    sortie = subprocess.run([sys.executable, "mesure_inference.py", mode, str(chemin), str(N_REPET), str(CONF),
                             json.dumps(images_bench)], capture_output=True, text=True)
    if sortie.returncode != 0:
        print(f"{mode} : échec\n", sortie.stderr[-1500:])
        continue
    mesures[mode] = json.loads(sortie.stdout.strip().splitlines()[-1])
    mesures[mode]["taille_mo"] = round(Path(chemin).stat().st_size / 1e6, 1)
    print(mode, "OK")


In [ ]:
tableau = pd.DataFrame(mesures).T[["taille_mo", "chargement_s", "inference_ms_mediane", "memoire_max_mo"]]
tableau.index = tableau.index.map({"pt": "best.pt (PyTorch)", "onnx": "model.onnx (Ultralytics)",
                                   "ort": "model.onnx (onnxruntime seul, comme CalEvent)"})
display(tableau)

# L'export a-t-il changé les détections ? (même seuil CONF, mêmes images)
if "pt" in mesures and "onnx" in mesures:
    for img, n_pt, n_onnx in zip(images_bench, mesures["pt"]["boites"], mesures["onnx"]["boites"]):
        marque = "" if n_pt == n_onnx else "  <-- différent"
        print(f"{Path(img).name:<30} best.pt : {n_pt:>3} boîtes | ONNX : {n_onnx:>3} boîtes{marque}")


**À noter dans vos livrables** (note de synthèse, partie déploiement) :

- Quel format est le plus **léger** ? le plus **rapide** ? lequel consomme le moins de **mémoire**, et pourquoi
  (indice : regardez ce que chaque processus doit importer) ?
- Les détections sont-elles identiques ? Un petit écart peut venir des arrondis de calcul ; un gros écart
  signale un problème d'export (taille d'image, version d'opset…).
- Si le serveur de déploiement avait 1 Go de mémoire libre, lequel choisiriez-vous ? Et pour une application
  **mobile** sans connexion ?
- Ces chiffres sont-ils valables sur la VM du cours ? Qu'est-ce qui changerait (nombre de cœurs, GPU absent) ?
